# Practical Functions for Data Analysis — pandas + NumPy

This notebook collects reusable, interview- and real-world-focused helper functions built with pandas and NumPy: data loading, cleaning, feature engineering, aggregations, time-series helpers, vectorized transforms, performance tips and small end-to-end examples.

## Setup: imports and sample data generators

In [1]:
import pandas as pd
import numpy as np
from typing import Optional
np.random.seed(0)

# small helper to create a sales-like DataFrame used in examples
def sample_sales(n_store=3, n_days=30):
    dates = pd.date_range('2020-01-01', periods=n_days, freq='D')
    rows = []
    for s in range(1, n_store+1):
        base = 100 + 20*s
        noise = np.random.randint(-10, 10, size=n_days)
        qty = np.clip(base + noise + np.arange(n_days) * 0.5, 0, None).astype(int)
        price = np.round(10 + 0.1*s + np.random.randn(n_days)*0.5, 2)
        for d, q, p in zip(dates, qty, price):
            rows.append({'store_id': s, 'date': d, 'units': int(q), 'price': float(p)})
    return pd.DataFrame(rows)

sales = sample_sales(4, 90)
sales.head()


,store_id,date,units,price
0,1,2020-01-01,122,10.85
1,1,2020-01-02,125,10.63
2,1,2020-01-03,111,9.76
3,1,2020-01-04,114,10.11
4,1,2020-01-05,115,9.91


## 1) I/O + lightweight validation helpers

In [13]:
def load_csv_checked(path: str, dtypes: Optional[dict] = None, parse_dates: Optional[list] = None) -> pd.DataFrame:
    """Load CSV with basic checks: shape, missing fraction, and memory summary.
    Returns DataFrame.
    """
    df = pd.read_csv(path, dtype=dtypes, parse_dates=parse_dates) if path else pd.DataFrame()
    print('loaded:', getattr(df, 'shape', None))
    if len(df) == 0:
        return df
    miss_frac = df.isna().mean().sort_values(ascending=False)
    print('top missing columns:', miss_frac[miss_frac>0].head())
    print('memory (MB):', df.memory_usage(deep=True).sum() / 1024**2)
    return df

# Example usage (commented because we don't have a file here):
#create a sample sales.csv file
sales.to_csv('sales.csv', index=False)
#load the file with datatype checks and date parsing
df = load_csv_checked('sales.csv', dtypes={'store_id':int}, parse_dates=['date'])


loaded: (360, 4)
top missing columns: Series([], dtype: float64)
memory (MB): 0.011112213134765625


## 2) Cleaning helpers: missing values, outliers, type fixes

In [3]:
def fill_missing_with_group_median(df: pd.DataFrame, group_cols: list, target_cols: list) -> pd.DataFrame:
    "Fill missing values in target_cols using group-level median."
    df = df.copy()
    for col in target_cols:
        df[col] = df.groupby(group_cols)[col].transform(lambda x: x.fillna(x.median()))
    return df

def cap_outliers_iqr(df: pd.DataFrame, cols: list, k: float = 1.5) -> pd.DataFrame:
    "Cap outliers using IQR rule (in-place returns copy)."
    df = df.copy()
    for c in cols:
        q1 = df[c].quantile(0.25)
        q3 = df[c].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        df[c] = df[c].clip(lower, upper)
    return df

# demo on `sales` DataFrame
s_demo = sales.copy()
s_demo.loc[0,'units'] = np.nan
print('missing before:', s_demo['units'].isna().sum())
s_demo = fill_missing_with_group_median(s_demo, ['store_id'], ['units'])
print('missing after:', s_demo['units'].isna().sum())
s_demo = cap_outliers_iqr(s_demo, ['units'])
s_demo['units'].describe()


missing before: 1
missing after: 0


count    360.000000
mean     171.594444
std       26.486491
min      111.000000
25%      151.750000
50%      172.000000
75%      192.000000
max      229.000000
Name: units, dtype: float64

## 3) Feature engineering helpers (date features, price features, grouping)

In [4]:
def engineer_date_features(df: pd.DataFrame, date_col: str='date') -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['day'] = df[date_col].dt.day
    df['weekday'] = df[date_col].dt.weekday
    df['month'] = df[date_col].dt.month
    df['is_weekend'] = df['weekday'] >= 5
    return df

def price_features(df: pd.DataFrame, price_col: str='price') -> pd.DataFrame:
    df = df.copy()
    df['log_price'] = np.log1p(df[price_col])
    df['price_z'] = (df[price_col] - df[price_col].mean()) / (df[price_col].std())
    return df

sales_fe = engineer_date_features(sales, 'date')
sales_fe = price_features(sales_fe, 'price')
sales_fe.head()


,store_id,date,units,price,day,weekday,month,is_weekend,log_price,price_z
0,1,2020-01-01,122,10.85,1,2,1,False,2.472328,1.237913
1,1,2020-01-02,125,10.63,2,3,1,False,2.453588,0.806501
2,1,2020-01-03,111,9.76,3,4,1,False,2.375836,-0.899538
3,1,2020-01-04,114,10.11,4,5,1,True,2.407846,-0.213200
4,1,2020-01-05,115,9.91,5,6,1,True,2.389680,-0.605393


## 4) Aggregation & KPIs: store-level and rolling metrics

In [5]:
def aggregate_store_daily(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    out = df.groupby(['store_id', pd.Grouper(key='date', freq='D')]).agg(
        units_sum=('units','sum'),
        units_mean=('units','mean'),
        revenue=('price', lambda s: (s * df.loc[s.index, 'units']).sum())
    ).reset_index()
    return out

agg = aggregate_store_daily(sales)
agg.head()


,store_id,date,units_sum,units_mean,revenue
0,1,2020-01-01,122,122.0,1323.70
1,1,2020-01-02,125,125.0,1328.75
2,1,2020-01-03,111,111.0,1083.36
3,1,2020-01-04,114,114.0,1152.54
4,1,2020-01-05,115,115.0,1139.65


In [6]:
def rolling_sales_metrics(df: pd.DataFrame, window: int = 7) -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['store_id','date'])
    df['units_rolling_mean'] = df.groupby('store_id')['units'].transform(lambda x: x.rolling(window, min_periods=1).mean())
    df['units_rolling_sum'] = df.groupby('store_id')['units'].transform(lambda x: x.rolling(window, min_periods=1).sum())
    return df

sales_roll = rolling_sales_metrics(sales, window=14)
sales_roll.head()


,store_id,date,units,price,units_rolling_mean,units_rolling_sum
0,1,2020-01-01,122,10.85,122.000000,122.0
1,1,2020-01-02,125,10.63,123.500000,247.0
2,1,2020-01-03,111,9.76,119.333333,358.0
3,1,2020-01-04,114,10.11,118.000000,472.0
4,1,2020-01-05,115,9.91,117.400000,587.0


## 5) Vectorized label/feature transforms and group operations

In [7]:
# vectorized conditional example: discount flag where price below percentile per store
def discount_flag_by_store(df: pd.DataFrame, pct: float = 0.2) -> pd.DataFrame:
    df = df.copy()
    thr = df.groupby('store_id')['price'].transform(lambda x: x.quantile(pct))
    df['is_discount'] = df['price'] <= thr
    return df

sales_disc = discount_flag_by_store(sales)
sales_disc.head()


,store_id,date,units,price,is_discount
0,1,2020-01-01,122,10.85,False
1,1,2020-01-02,125,10.63,False
2,1,2020-01-03,111,9.76,False
3,1,2020-01-04,114,10.11,False
4,1,2020-01-05,115,9.91,False


## 6) Real-world mini-example: Top-3 products per customer (user events) and sessionization

In [8]:
# sample events dataset
def sample_events(n_users=5, n_events=100):
    users = np.random.choice(range(1,n_users+1), size=n_events)
    items = np.random.choice(range(100,120), size=n_events)
    times = pd.to_datetime('2020-01-01') + pd.to_timedelta(np.random.randint(0, 60*24, size=n_events), unit='m')
    scores = np.random.randint(1,10, size=n_events)
    return pd.DataFrame({'user_id':users, 'item_id':items, 'ts':times, 'score':scores})

events = sample_events(10, 200)
# top-3 items per user by score (groupby + nlargest)
topk = events.groupby('user_id', group_keys=False).apply(lambda d: d.nlargest(3, 'score'))
topk.head()

# sessionization: group consecutive events within 30 minutes per user
events = events.sort_values(['user_id','ts'])
events['prev_ts'] = events.groupby('user_id')['ts'].shift(1)
events['gap_min'] = (events['ts'] - events['prev_ts']).dt.total_seconds().div(60)
events['new_session'] = (events['gap_min'] > 30) | events['gap_min'].isna()
events['session_id'] = events.groupby('user_id')['new_session'].cumsum()
events.head()


,user_id,item_id,ts,score,prev_ts,gap_min,new_session,session_id
143,1,109,2020-01-01 00:05:00,7,NaT,NaN,True,1
142,1,108,2020-01-01 00:55:00,2,2020-01-01 00:05:00,50.0,True,2
98,1,113,2020-01-01 01:42:00,2,2020-01-01 00:55:00,47.0,True,3
64,1,102,2020-01-01 02:31:00,7,2020-01-01 01:42:00,49.0,True,4
37,1,105,2020-01-01 02:39:00,4,2020-01-01 02:31:00,8.0,False,4


## 7) Reusable pipelines & `pipe` pattern

In [9]:
def preprocess_sales_pipeline(df):
    return (df.pipe(engineer_date_features, 'date')
              .pipe(fill_missing_with_group_median, ['store_id'], ['units'])
              .pipe(price_features, 'price')
              .pipe(rolling_sales_metrics, 7))

sales_pre = preprocess_sales_pipeline(sales)
sales_pre.head()


,store_id,date,units,price,day,weekday,month,is_weekend,log_price,price_z,units_rolling_mean,units_rolling_sum
0,1,2020-01-01,122,10.85,1,2,1,False,2.472328,1.237913,122.000000,122.0
1,1,2020-01-02,125,10.63,2,3,1,False,2.453588,0.806501,123.500000,247.0
2,1,2020-01-03,111,9.76,3,4,1,False,2.375836,-0.899538,119.333333,358.0
3,1,2020-01-04,114,10.11,4,5,1,True,2.407846,-0.213200,118.000000,472.0
4,1,2020-01-05,115,9.91,5,6,1,True,2.389680,-0.605393,117.400000,587.0


## 8) Performance tips and alternatives (NumPy vectorization, chunked reads, dtypes)

In [10]:
# vectorized example: compute revenue column without explicit Python loops
sales_vec = sales.copy()
sales_vec['revenue'] = sales_vec['units'].to_numpy(dtype=float) * sales_vec['price'].to_numpy(dtype=float)
print(sales_vec.head())

# chunked read example (psuedocode):
# for chunk in pd.read_csv('big.csv', chunksize=100_000):
#     process(chunk)

# dtype example: reduce memory by using categorical for low-cardinality columns
print('before dtype:', sales.memory_usage(deep=True).sum())
sales['store_id'] = sales['store_id'].astype('int8')
print('after dtype:', sales.memory_usage(deep=True).sum())


   store_id       date  units  price  revenue
0         1 2020-01-01    122  10.85  1323.70
1         1 2020-01-02    125  10.63  1328.75
2         1 2020-01-03    111   9.76  1083.36
3         1 2020-01-04    114  10.11  1152.54
4         1 2020-01-05    115   9.91  1139.65
before dtype: 11652
after dtype: 9132


## 9) Small utilities: save summary, top-n function, merge-by-recent

In [11]:
def top_n_by_group(df: pd.DataFrame, group_col: str, sort_col: str, n: int = 3) -> pd.DataFrame:
    return df.groupby(group_col, group_keys=False).apply(lambda d: d.nlargest(n, sort_col))

def merge_keep_latest(left: pd.DataFrame, right: pd.DataFrame, on: str, time_col: str) -> pd.DataFrame:
    # keep latest right-row per key before merging (common in enrichment tasks)
    r = right.sort_values(time_col).drop_duplicates(on=on, keep='last')
    return left.merge(r, on=on, how='left')

# demo top_n_by_group on events
print(top_n_by_group(events, 'user_id', 'score', 2).head())


     item_id                  ts  score             prev_ts  gap_min  \
50       119 2020-01-01 03:13:00      9 2020-01-01 02:39:00     34.0   
159      102 2020-01-01 06:02:00      8 2020-01-01 03:34:00    148.0   
115      104 2020-01-01 01:06:00      9 2020-01-01 01:03:00      3.0   
146      112 2020-01-01 17:53:00      9 2020-01-01 17:49:00      4.0   
30       104 2020-01-01 09:58:00      9 2020-01-01 07:59:00    119.0   

     new_session  session_id  
50          True           5  
159         True           6  
115        False           1  
146        False           6  
30          True           4  


## Exercises & next steps

- Convert the preprocessing pipeline into a small CLI script that accepts an input CSV and writes features to Parquet.
- Add tests for each helper function (unit tests using `pytest`).
- Expand with ML-ready transforms (scikit-learn `ColumnTransformer`, imputer + scaler).
- I can also run these cells here to verify output if you want — tell me to run them.